To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://unsloth.ai/docs/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth on your local device, follow [our guide](https://unsloth.ai/docs/get-started/install). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & how to save it.

This notebook fine-tunes **[SmolLM3-3B](https://huggingface.co/HuggingFaceTB/SmolLM3-3B)** (Hugging Face's small hybrid-reasoning model) for conversational chat.

### News

Unsloth now supports **SmolLM3**! SmolLM3-3B is a small (3B) hybrid model that can run in a fast **`/no_think`** mode or a deliberate **`/think`** reasoning mode. This notebook trains the fast conversational (`/no_think`) mode so it fits a free Tesla T4.

Read our docs for the latest models & guides: [unsloth.ai/docs](https://unsloth.ai/docs).

### Installation

In [ ]:
%%capture
# On Colab, a plain install is all you need and pulls a matching Unsloth + Transformers set.
!pip install unsloth

<a name="Unsloth"></a>
### Unsloth

In [ ]:
from unsloth import FastModel
import torch
max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = None           # None for auto detection. Float16 for Tesla T4, Bfloat16 for Ampere+
load_in_4bit = True    # 4bit quantization to reduce memory. Can be False.

# 4bit pre-quantized models we support for 4x faster downloading + no OOMs:
fourbit_models = [
    "unsloth/SmolLM3-3B",                       # This notebook's model
    "unsloth/Qwen3-4B-Instruct-2507",
    "unsloth/gemma-3-4b-it",
    "unsloth/Phi-4",
]  # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/SmolLM3-3B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    full_finetuning = False,
)

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,  # text-only model
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 16,  # Choose any number > 0! Suggested 8, 16, 32, 64, 128
    lora_alpha = 16,
    lora_dropout = 0,   # Supports any, but = 0 is optimized
    bias = "none",      # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth",  # 30% less VRAM, fits 2x larger batch sizes
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep
We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style, and convert it to HuggingFace's `("role", "content")` format.

Note on SmolLM3: its chat template defaults to `/think` reasoning mode, which adds a reasoning system prompt and expects `<think>...</think>` sections. FineTome has no reasoning traces, so we set `enable_thinking=False` (`/no_think`) so the template matches the data. In `/no_think` mode SmolLM3 still emits an empty `<think></think>` block before the answer, which is expected, so we keep it. We use SmolLM3's own chat template and don't override it with `get_chat_template`.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt

dataset = load_dataset("mlabonne/FineTome-100k", split = "train")
dataset = standardize_sharegpt(dataset)  # -> {"role", "content"} format

def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize = False, add_generation_prompt = False,
            enable_thinking = False,   # SmolLM3: fast /no_think mode, matches non-reasoning data
        )
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)

We look at how the conversations are structured for item 5:

In [ ]:
dataset[5]["conversations"]

And we see how SmolLM3's chat template transformed these conversations (note the short `/no_think` system prompt and the intentional empty `<think></think>` block before the assistant's answer):

In [ ]:
dataset[5]["text"]

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support `DPOTrainer` and `GRPOTrainer` for reinforcement learning!

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForSeq2Seq

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False,  # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        # num_train_epochs = 1,  # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",  # Use TrackIO/WandB etc
    ),
)

We use Unsloth's `train_on_responses_only` to train only on the assistant's replies and ignore loss on the user's inputs. Unsloth auto-detects SmolLM3's ChatML instruction/response markers from the chat template.

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(trainer)

We verify masking is done — only the assistant's reply should be unmasked (system + user show as blank):

In [ ]:
space = tokenizer(" ", add_special_tokens = False).input_ids[0]
tokenizer.decode([space if x == -100 else x for x in trainer.train_dataset[5]["labels"]])

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
trainer_stats = trainer.train()

<a name="Inference"></a>
### Inference
Let's run the model! We keep `/no_think` (`enable_thinking=False`) to match how we trained it. We pass the full tokenized dict so `attention_mask` is set (avoids the pad==eos warning).

In [ ]:
messages = [{"role": "user", "content": "Explain what a black hole is, in simple terms."}]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    enable_thinking = False,
    return_tensors = "pt",
    return_dict = True,       # returns input_ids + attention_mask
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 256,
    temperature = 0.7,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<a name="Save"></a>
### Saving to float16 for VLLM / GGUF
We save the LoRA adapters below. To merge to 16-bit or export GGUF (for llama.cpp / Ollama), uncomment the relevant lines.

In [ ]:
# Save the LoRA adapters (small)
model.save_pretrained("smollm3-finetome-lora")
tokenizer.save_pretrained("smollm3-finetome-lora")

# Merge to 16-bit (for vLLM / HF deployment):
# model.save_pretrained_merged("smollm3-finetome-merged", tokenizer, save_method = "merged_16bit")

# Export to GGUF for llama.cpp / Ollama:
# model.save_pretrained_gguf("smollm3-finetome-gguf", tokenizer, quantization_method = "q4_k_m")

And we're done! If you have any questions, join our [Discord](https://discord.gg/unsloth). Some other links:
- ⭐ Star Unsloth on [GitHub](https://github.com/unslothai/unsloth)
- 📚 [Unsloth docs](https://unsloth.ai/docs) for all our models & guides